# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)




## 1. My lane as an ML task (type)

**Type: a scoring / ranking task, built on top of a binary classification model.**

I need a **classification/probability model** — something that outputs a probability that a page is a good review candidate (declining, thin, low-CTR, low-engagement, etc.), the same way the starter pipeline outputs a `best_model_probability` from logistic regression / decision tree / random forest.

On top of that, the probability gets blended with a transparent baseline score into a **final score**, and pages get **sorted** by that score. Sorting-by-score-for-limited-capacity is what makes this a ranking task rather than a plain classification task.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Starter (proxy) target — what I'll use first:**

```
is_declining_label = trend_direction == "down"
```

This label comes from a **defined rule**, not a future observed outcome: `trend_direction` is a bucket already computed from the current 90-day window in the data. It's useful for getting the whole pipeline working honestly, but per the guide it's a beginner proxy — it describes "is this page currently bucketed as down," not "will this page actually decline further."

**Stronger (capstone-direction) target — what I want to earn my way to:**

```
features from prior 90 days -> decline OR recovery over the next 30 days
```

This is an **observed future outcome**: define a feature window (e.g. days 1-90) and a separate, non-overlapping target window (days 91-120), then label a page as "declined" if a chosen metric (impressions, clicks, or sessions) drops past a magnitude+persistence threshold I write down in advance, or "recovered" under the mirrored condition. This is strictly better evidence than the proxy because it can't be circular — the label is measured after the decision point, from data the model never saw during feature construction.

Either way, the label is something the data can prove happened (or a rule I state plainly as a rule) — never one of FlyRank's own product decision fields (`health_score`, `priority_score`, `action_type`), which aren't even shipped in this dataset and would be circular if they were.

In [3]:
# Section 2: define both the starter proxy label and the stronger future-window label
# (implemented here on the starter CSV; the future-window version needs the daily warehouse
# table to build feature/target windows properly -- flagged as a TODO for the warehouse pass).

import pandas as pd
from pathlib import Path

csv_path = Path("content_refresh_anonymized.csv")
if csv_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv -- run from the repo "
        "(the Colab badge above clones it), or fix CSV_CANDIDATES."
    )

raw = pd.read_csv(csv_path)
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset="content_id")

# Starter proxy label -- a rule on the current window, not a future outcome.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows in scope: {len(df):,}")
print(f"Positive rate (is_declining_label == 1): {df['is_declining_label'].mean():.1%}")
print(
    "NOTE: stronger target (decline/recovery over the NEXT 30 days, built from the "
    "PRIOR 90 days) requires fact_content_daily_performance with an explicit, "
    "non-overlapping feature/target window -- not available in this starter CSV, "
    "which is a single already-aggregated snapshot."
)


Rows in scope: 30,000
Positive rate (is_declining_label == 1): 54.2%
NOTE: stronger target (decline/recovery over the NEXT 30 days, built from the PRIOR 90 days) requires fact_content_daily_performance with an explicit, non-overlapping feature/target window -- not available in this starter CSV, which is a single already-aggregated snapshot.


## 3. Success metric

**Metric: Precision@50** (with Average Precision and Recall tracked alongside, not instead).

Why this one, and not plain accuracy or ROC AUC alone:

- The reviewer's real constraint is **capacity**, not correctness-in-general — they will look at roughly the top 50 pages this cycle, full stop. A metric that doesn't respect that (accuracy, or AUC over the whole ranking) can look great while the top 50 are mediocre.
- **Precision@50** answers the exact operational question: *of the top 50 pages I hand the reviewer, how many are genuinely worth their time?* That's a number I can defend to a non-technical stakeholder in one sentence.
- I'll track **Average Precision** too, since it rewards getting the ordering right across the whole ranked list, not just the cutoff — useful for comparing models even if the team's capacity changes later.
- I'll track **Recall** as a guardrail, because the guide is clear that under fixed capacity, a missed true problem is often the more expensive failure mode than a wasted review slot; if a model pushes Precision@50 up only by badly starving recall, that's not actually a win.
- "Good" in numbers, using the starter's own verified results as the bar to beat: baseline rules score **0.240** Precision@50, so I'd call a model 'good' if it clears roughly **0.50+** on this lane's slice, and 'strong' near the starter's own random forest result of **0.740** — while remembering that number came from a 30k-row anonymized slice with client-holdout validation, not the full warehouse, so it has to be earned again on whatever data I actually validate against.

In [4]:
# Section 3: show what Precision@K actually means on this slice, using the
# starter's own already-verified numbers as the reference point (no re-training here --
# just making the metric concrete against real figures from outputs/model_report.md).

starter_results = pd.DataFrame(
    {
        "method": ["baseline rules", "logistic regression", "decision tree", "random forest"],
        "roc_auc": [0.627, 0.700, 0.742, 0.750],
        "average_precision": [0.468, 0.522, 0.575, 0.618],
        "precision_at_50": [0.240, 0.400, 0.540, 0.740],
    }
)
starter_results["pages_right_of_top_50"] = (starter_results["precision_at_50"] * 50).round().astype(int)
print(starter_results.to_string(index=False))

print()
print(f"Positive rate in current scope: {df['is_declining_label'].mean():.1%}  "
      f"(the metric that matters isn't this base rate, it's precision within the reviewer's top-K)")


             method  roc_auc  average_precision  precision_at_50  pages_right_of_top_50
     baseline rules    0.627              0.468             0.24                     12
logistic regression    0.700              0.522             0.40                     20
      decision tree    0.742              0.575             0.54                     27
      random forest    0.750              0.618             0.74                     37

Positive rate in current scope: 54.2%  (the metric that matters isn't this base rate, it's precision within the reviewer's top-K)


## 4. The unit of analysis, as a real dataframe

**One row = one content item (`content_id`), for one client (`client_id`), summarized over a fixed 90-day window.**

Not one row per day, not one row per query — the daily fact table (`fact_content_daily_performance`) is a time series I roll up into one snapshot row per page before scoring, exactly like the starter pipeline's `01_prepare_features.py` does. Below I load that unit of analysis directly from the starter CSV and show it.

In [5]:
# Section 4: the unit of analysis, as a real dataframe -- one row = one content item.

unit_cols = [
    c for c in [
        "content_id", "client_id", "impressions_90d", "sessions_90d",
        "content_age_days", "trend_direction", "avg_position", "word_count",
        "days_since_last_update",
    ]
    if c in df.columns
]

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content_id count: {df['content_id'].nunique():,}  (should equal row count -- one row per page)")
assert df["content_id"].nunique() == len(df), "Expected exactly one row per content_id after dedup."

df[unit_cols].head(10)


Shape: 30,000 rows x 45 columns
Unique content_id count: 30,000  (should equal row count -- one row per page)


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,avg_position,word_count,days_since_last_update
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,10.6,3221.0,20
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,20.3,2481.0,25
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,36.5,3515.0,20
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,6.2,NaN,22
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,44.0,2803.0,14
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,8.5,3080.0,20
6,content_9a34b442b552,client_8722616204,20,1,90,down,7.0,3059.0,20
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,21.2,NaN,22
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,46.0,3807.0,20
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,4.9,NaN,104


## 5. Why ML beats a fixed rule here

**Why an if-statement isn't enough:**

- **The signals interact, they don't just add up.** A page with high impressions but a strong position can be fine; the same impressions with a weak position and a stale update date is a different story. A hand-written rule like the starter baseline (`0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap`) has to guess fixed weights and thresholds up front — it can't learn that, say, freshness matters twice as much once position is already borderline. A model can pick up on that kind of conditional, non-linear structure from data instead of a guess.
- **The thresholds themselves are policy choices, not universal truths** (the guide says this outright in section 11) — "180 days stale," "500 impressions," "CTR < 0.5" are reasonable starting rules, but nothing says they're the *right* cut points for this data. A model effectively learns better cut points (and combinations of them) from what actually happened to pages historically, instead of me guessing six numbers by hand.
- **There's already measured evidence a fixed rule underperforms here.** Per the starter's own verified results (`outputs/model_report.md`), the rule-based baseline gets **Precision@50 = 0.240** (~12 of the top 50 right) while a random forest on the same slice gets **0.740** (~37 of the top 50 right). That's not a hypothetical — it's the exact gap this lane exists to close.
- **What a rule *is* still good for:** transparency and reason codes. I'm not throwing the baseline away — I'm using it as the honest floor to beat, and its individual conditions (`stale_visible_page`, `thin_visible_page`, etc.) become the human-readable reason codes attached to the model's ranked output, so a reviewer never just sees an opaque number.

In [6]:
# Section 5: quick, honest look at whether a single-threshold rule cleanly separates
# the label -- if declining and non-declining pages overlap heavily on a "simple" signal,
# that's direct evidence a fixed if-statement isn't enough.

import numpy as np

if {"avg_position", "days_since_last_update"}.issubset(df.columns):
    overlap = df.groupby("is_declining_label")[["avg_position", "days_since_last_update"]].describe(
        percentiles=[0.25, 0.5, 0.75]
    )
    print(overlap)
    print()
    print(
        "If the 25th-75th percentile ranges for declining vs. non-declining pages "
        "overlap heavily on these signals individually, no single threshold on either "
        "one alone will cleanly separate the classes -- which is the concrete evidence "
        "for needing a model that can combine signals, not just one more if-statement."
    )
else:
    print("avg_position / days_since_last_update not present in this slice -- "
          "substitute whatever signal columns this CSV actually ships to run this check.")


                   avg_position                                               \
                          count       mean        std  min  25%    50%   75%   
is_declining_label                                                             
0                       13738.0  16.823060  17.327709  0.0  5.5  10.05  23.4   
1                       16262.0  15.936305  13.159382  0.0  6.7  11.30  21.6   

                          days_since_last_update                             \
                      max                  count       mean        std  min   
is_declining_label                                                            
0                   245.0                13738.0  42.372543  40.857322  1.0   
1                   118.0                16262.0  49.245788  42.832998  1.0   

                                              
                     25%   50%    75%    max  
is_declining_label                            
0                   20.0  20.0  104.0  373.0  
1              

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.